# Breast Cancer - Pré-processamento

Transforma o `Breast_Cancer.csv` bruto em uma tabela 100% numérica, pronta para os modelos.
Os motivos de cada passo estão em [DICIONARIO_DE_DADOS.md](../dataset/DICIONARIO_DE_DADOS.md).

In [ ]:
import pandas as pd

df = pd.read_csv("../dataset/Breast_Cancer.csv")
print(df.shape)

## 1. Limpeza de texto

Remove espaços sobrando nos nomes das colunas (`"T Stage "`) e nos valores (`"Single "`, `" anaplastic; Grade IV"`).
Também corrige o erro de digitação `Reginol` → `Regional`.

In [ ]:
df.columns = df.columns.str.strip()
df = df.rename(columns={"Reginol Node Positive": "Regional Node Positive"})

colunas_texto = df.select_dtypes(include="str").columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.str.strip())

## 2. Duplicados

In [ ]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

## 3. Remoção de colunas

- `differentiate`: mesma informação de `Grade`.
- `6th Stage`: é derivada de `T Stage` + `N Stage`.
- `Survival Months`: vazamento de dados (está ligada diretamente ao `Status`).

In [ ]:
df = df.drop(columns=["differentiate", "6th Stage", "Survival Months"])

## 4. Codificação

- **Ordinais** (têm ordem de gravidade): viram números inteiros.
- **Binárias**: viram 0/1.
- **Nominais** (sem ordem): one-hot encoding.

In [ ]:
ordinais = {
    "T Stage": {"T1": 1, "T2": 2, "T3": 3, "T4": 4},
    "N Stage": {"N1": 1, "N2": 2, "N3": 3},
    "Grade": {"1": 1, "2": 2, "3": 3, "anaplastic; Grade IV": 4},
}
binarias = {
    "A Stage": {"Regional": 0, "Distant": 1},
    "Estrogen Status": {"Negative": 0, "Positive": 1},
    "Progesterone Status": {"Negative": 0, "Positive": 1},
    "Status": {"Alive": 0, "Dead": 1},  # alvo: 1 = óbito
}

for coluna, mapa in {**ordinais, **binarias}.items():
    df[coluna] = df[coluna].map(mapa)

df = pd.get_dummies(df, columns=["Race", "Marital Status"], dtype=int)

## 5. Verificação

In [ ]:
# Se algum valor não estava no mapa, ele vira NaN
assert df.isna().sum().sum() == 0, "Há valores que não foram mapeados"

print(df.shape)
print(df["Status"].value_counts(normalize=True).round(3))
df.head()

## 6. Salvar

A padronização (scaler) e a divisão treino/teste ficam para a etapa de modelagem:
o scaler precisa ser ajustado só com os dados de treino para não vazar informação do teste.

In [ ]:
df.to_csv("../dataset/Breast_Cancer_processed.csv", index=False)